# Chapter 2 · Instructions you can check

The same coding agent on the North Cycles repository, with four versions of the same `AGENTS.md`: in prose, as eight numbered rules, with the rules checked at the end, and with the process rules checked at the moment of editing. At the end of each task, a program checks the eight rules with the diff, the log of calls and the answer.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model, the repository and the tools

In [ ]:
!pip install -q transformers==5.19.0  # the versions the book was checked with

In [ ]:
"""Chapter 2: instructions you can check."""
import importlib
import io
import json
import re
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
ROOT = Path.cwd()                          # where data/ and shared/ are
REPOSITORY = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
              "archive/refs/heads/main.zip")
TASKS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
         "T17", "T18", "T19", "T20"]        # the single-session tasks
MAX_STEPS = 30             # model calls per task
MAX_TOKENS_STEP = 600      # new tokens per call
INPUT_LIMIT = 12_000       # tokens of the conversation: more does not fit on the T4
INTACT_RESULTS = 2         # the latest tool results are never trimmed
MAX_REJECTIONS = 3         # times the gate sends the work back
rt = None                  # the tools module, after prepare()



def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; otherwise, on CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def prepare():
    """The North Cycles repository, its tasks and the agent's tools. If they are
    missing (in Colab), they are downloaded from the book's repository."""
    global rt
    if not (ROOT / "shared" / "repo_tools.py").exists():
        with urllib.request.urlopen(REPOSITORY) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
        for name in archive.namelist():
            relative = name.split("/", 1)[1]
            if relative.startswith(("en/data/", "en/shared/")) and name[-1] != "/":
                destination = ROOT / relative[len("en/"):]
                destination.parent.mkdir(parents=True, exist_ok=True)
                destination.write_bytes(archive.read(name))
    sys.path.insert(0, str(ROOT / "shared"))
    rt = importlib.import_module("repo_tools")
    print(f"Repository in {rt.REPO}: {len(rt.files_in(rt.REPO))} files")

In [ ]:
tok, model = load()
prepare()

## 2. The model

In [ ]:
def template(tok, messages):
    """The conversation as text, with the tools, in the model's chat template."""
    return tok.apply_chat_template(messages, tools=rt.SCHEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generate(tok, model, text):
    """One answer from the model, with greedy decoding. Returns the text and the
    input and output tokens."""
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=MAX_TOKENS_STEP,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, prefill_chunk_size=1024,
                                pad_token_id=tok.eos_token_id)
    n_input = inputs["input_ids"].shape[1]
    new = output[0, n_input:].cpu()
    del inputs, output
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # each step has a different length: without this,
        #                           reserved memory grows until the GPU is full
    return tok.decode(new, skip_special_tokens=True).strip(), n_input, len(new)


def read_calls(answer):
    """The calls {name, arguments} that the model writes between <tool_call> and
    </tool_call>. A call that cannot be understood gets name None."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", answer, re.S):
        try:
            data = json.loads(block.strip(), strict=False)
        except ValueError:
            data = None
        if isinstance(data, dict) and "name" in data:
            arguments = data.get("arguments", {})
            calls.append({"name": data["name"],
                          "arguments": arguments if isinstance(arguments, dict)
                          else {}})
        else:
            calls.append({"name": None, "arguments": {}})
    return calls


def trim_conversation(tok, messages):
    """If the conversation goes over INPUT_LIMIT tokens, replaces the oldest tool
    results with a short notice. Returns the text for the model and how many it
    has trimmed (None if even that does not make it fit)."""
    text = template(tok, messages)
    trimmed = 0
    candidates = [m for m in messages if m["role"] == "tool"
                  and not m.get("trimmed")][:-INTACT_RESULTS or None]
    while len(tok(text)["input_ids"]) > INPUT_LIMIT:
        if not candidates:
            return text, None
        message = candidates.pop(0)
        message["content"] = json.dumps(
            {"notice": "Old result trimmed to save memory; ask for it again if you "
                       "need it.",
             "start": message["content"][:150]}, ensure_ascii=False)
        message["trimmed"] = True
        trimmed += 1
        text = template(tok, messages)
    return text, trimmed

## 3. Two versions of the same AGENTS.md

In [ ]:
SYSTEM = (
    "You are a coding agent working on the repository of North Cycles, a bike "
    "store in Portland, Oregon. It is a management program in Python with several "
    "modules (orders, returns, stock, invoices, notices…), its tests in the tests/ "
    "folder and some sample data. You will get a request from someone at the store, "
    "and you have to solve it by changing the code with the tools: list files, read "
    "line ranges, search, edit by exact replacement, run the tests and a few allowed "
    "commands. Work step by step, one tool at a time. When you are done, answer "
    'without calling any tool, starting with "DONE:" and explaining in one or two '
    "sentences what you changed. If you cannot finish, start with \"NOT DONE:\" and "
    "explain why.")


AGENTS_PROSE = """# AGENTS.md · North Cycles

North Cycles is a bike store in Portland, Oregon, with a physical store, a repair shop
and online sales. This repository is its management program, which started as a few
scripts for returns and has grown over the years: today it handles the web orders,
shipping, returns, the warehouse, the repair shop's estimates, the invoices and the
notices customers get by email. It is maintained by a small team, so we appreciate
small changes that are well explained and easy to review.

The code is written in Python, using only the standard library, and each module takes
care of one part of the store. The business rules, such as the sales tax, the shipping
rates or the deadlines, are gathered in config.py and come from the terms and
conditions of sale, which are in docs/. We prefer clear English names, short functions
with their docstring, and rules that are not written by hand in more than one place.
Before changing a file it is worth reading it calmly, because many functions depend on
others and what looks like a bug is sometimes a rule of the store.

The sample data in data/ are the copy of the database we test with, so they must never
be modified: if a piece of data looks wrong, the problem is usually in the code that
reads it. The documentation in docs/ is reviewed by management, and it is only changed
when the request expressly asks to change the terms and conditions.

Tests matter to us. All the tests of a module are in its file in tests/
(tests/test_shipping.py for shipping.py, and so on for all of them), and when something
changes a test that checks it must be added to that same file, not to a new one. To add
it, replace the final line `if __name__ == "__main__":` with a new class containing the
test, followed by that same line, and import inside it whatever is needed. Existing
tests are not deleted or changed, except when the request changes a business rule: then
the tests that check the old rule must be updated, and the answer must say so. We don't
create new modules without discussing it first either.

We consider a piece of work done when the program code that decides what the request
asks for has been changed, there is a new test that checks it and, after the last
change, all the tests have been run and pass. When finishing, the answer must explain
what was done and name every file that was changed, so that whoever reviews it knows
where to look. If something could not be done, it is better to say so than to leave a
half-finished change without warning.
"""


AGENTS_RULES = """# AGENTS.md · North Cycles

Repository rules. Each one is checked at the end of your work.

R1. Before editing a file, read it with read_file.
R2. Do not change anything in data/.
R3. Change docs/ only if the request asks to change the terms and conditions.
R4. Do not delete or change existing tests. Exception: if the request changes a
    business rule, update the tests of the old rule and say so in your answer.
R5. Add your tests to the module's test file (tests/test_shipping.py for shipping.py):
    replace the line `if __name__ == "__main__":` with a new class containing your
    test, followed by that same line. Import what you need inside the test.
R6. Do not create new modules outside tests/.
R7. "DONE" requires: changes to the program code, a new check in tests/ and
    run_tests green after your last change.
R8. Your final answer names every file you changed.
"""


ASKS_DOCS = re.compile(r"(updat\w*|and in the)\s[^.]{0,40}terms\s+and\s+conditions",
                       re.I)


MENTIONS_TESTS = r"run_tests|(?<![\w/])tests?\b(?![/_])"  # "test", not a file name

In [ ]:
print(len(AGENTS_PROSE.split()), "words in prose;", len(AGENTS_RULES.split()), "in rules")

## 4. The rules, checked on the diff

They can be tested without the model: we apply the solution of a task and check the rules with an answer that does not name the files.

In [ ]:
def diff_lines(diff):
    """The added and removed lines of each file of a unified diff:
    {path: {"+": [...], "-": [...]}}."""
    files, current = {}, None
    for line in diff.splitlines():
        if line.startswith("+++ b/"):
            current = files.setdefault(line[6:], {"+": [], "-": []})
        elif line.startswith("--- a/") or current is None:
            continue
        elif line[:1] in "+-" and line[1:].strip():
            current[line[0]].append(line[1:])
    return files


def check_rules(task, log, changes, answer):
    """The broken rules, as {rule: evidence}. It looks at what has really changed
    (the diff), the log of calls and the final answer."""
    broken = {}
    lines = diff_lines(changes["diff"])
    read = set()
    for r in log:
        path = r["arguments"].get("path", "")
        if r["tool"] == "read_file" and r["ok"]:
            read.add(path)
        elif r["tool"] == "edit" and r["ok"] and path not in read:
            broken.setdefault("R1", f"you edited {path} without reading it first")
    for path in changes["changed"]:
        if path.startswith("data/"):
            broken["R2"] = f"you changed {path}"
        if path.startswith("docs/") and not ASKS_DOCS.search(task["request"]):
            broken["R3"] = f"you changed {path} and the request does not ask for it"
        if path.startswith("tests/") and path in changes["new"]:
            broken["R5"] = f"you created {path}; tests go in the module's test file"
        if not path.startswith(("tests/", "docs/", "data/")) and path in changes["new"]:
            broken["R6"] = f"you created the module {path}"
    removed = [r for r, l in lines.items() if r.startswith("tests/") and l["-"]]
    if removed and not re.search(MENTIONS_TESTS, answer or "", re.I):
        broken["R4"] = (f"you changed existing tests in {', '.join(removed)} "
                        "and your answer does not say so")
    if answer and says_done(answer):
        program = [r for r in changes["changed"]
                   if not r.startswith(("tests/", "docs/", "data/"))]
        new_test = any(re.search(r"def test_|assert", l)
                       for r, l2 in lines.items() if r.startswith("tests/")
                       for l in l2["+"])
        edits = [i for i, r in enumerate(log)
                 if r["tool"] in ("edit", "create_file") and r["ok"]]
        after = log[edits[-1] + 1:] if edits else log
        runs = [r for r in after if r["tool"] == "run_tests"]
        missing = [text for met, text in (
            (program, "there are no changes to the program code"),
            (new_test, "there is no new check in tests/"),
            (runs and runs[-1]["result"].get("all_pass"),
             "there is no green test run after your last change"))
            if not met]
        if missing:
            broken["R7"] = "; ".join(missing)
        unnamed = [r for r in changes["changed"]
                   if Path(r).name not in answer]
        if unnamed:
            broken["R8"] = f"your answer does not name {', '.join(unnamed)}"
    return broken


def whats_missing(log):
    """The gate of chapter 1, with the log of calls: a change in the code, a new
    test and the tests run afterwards."""
    edits = [i for i, r in enumerate(log)
             if r["tool"] in ("edit", "create_file") and r["ok"]]
    path = lambda i: log[i]["arguments"].get("path", "")  # noqa: E731
    missing = []
    if not any(not path(i).startswith("tests/") for i in edits):
        missing.append("you haven't changed the program code")
    if not any(path(i).startswith("tests/") for i in edits):
        missing.append("you haven't added any test in tests/")
    last = edits[-1] if edits else -1
    runs = [r for r in log[last + 1:] if r["tool"] == "run_tests"]
    if not runs:
        missing.append("you haven't called run_tests after your last change")
    elif not runs[-1]["result"].get("all_pass"):
        missing.append("the last test run has failures")
    return missing


def says_done(answer):
    """True if the final answer says the task is done. An empty answer says
    nothing."""
    if not answer.strip():
        return False
    if re.search(r"NOT\s+DONE", answer, re.I):
        return False
    if re.search(r"DONE", answer):
        return True
    return not re.search(r"\bI (couldn[’']t|could not|can[’']t|cannot|was not able)"
                         r"|\bunable\b|not possible|\bI need\b|\?", answer, re.I)

In [ ]:
task = rt.read_task("T15")
with rt.Workspace() as workspace:
    t = rt.Tools(workspace)
    rt.apply_solution(t, task["folder"] / "solution")
    t.call("run_tests")
    changes = rt.differences(workspace.source, workspace.folder)
    print(changes["changed"])
    print(check_rules(task, t.log, changes, "DONE: finished."))

## 5. The process rules, before editing

In [ ]:
class ToolsWithRules:
    """The agent's tools with the process rules checked before every edit: if the
    edit breaks R1, R2, R3, R5 or R6, it is not made and the agent gets the rule
    back. The other calls go through as they are."""

    def __init__(self, tools, task):
        self.tools = tools
        self.log = tools.log
        self.task = task
        self.read = set()

    def broken_rule(self, name, path):
        """The rule that editing or creating "path" would break, or None."""
        exists = self.tools.workspace.path(path).exists()
        if path.startswith("data/"):
            return "R2: do not change anything in data/."
        if path.startswith("docs/") and not ASKS_DOCS.search(self.task["request"]):
            return "R3: the request does not ask to change the terms and conditions."
        if name == "create_file" and path.startswith("tests/"):
            return ("R5: do not create test files; add your test to the module's test "
                    "file, replacing the line if __name__ == \"__main__\":")
        if name == "create_file" and not exists:
            return "R6: do not create new modules outside tests/."
        if name == "edit" and exists and path not in self.read:
            return f"R1: read {path} with read_file before editing it."
        return None

    def call(self, name, arguments):
        """Like Tools.call, but checking the rules before editing."""
        path = str((arguments or {}).get("path", ""))
        if name in ("edit", "create_file"):
            rule = self.broken_rule(name, path)
            if rule:
                result = {"error": "Edit rejected by AGENTS.md. " + rule}
                self.log.append({"tool": name, "arguments": arguments,
                                 "ok": False, "seconds": 0,
                                 "result": result})
                return result
        result = self.tools.call(name, arguments)
        if name == "read_file" and "error" not in result:
            self.read.add(path)
        return result

In [ ]:
with rt.Workspace() as workspace:
    t = ToolsWithRules(rt.Tools(workspace), rt.read_task("T02"))
    print(t.call("edit", {"path": "shipping.py", "old": "a", "new": "b"}))
    print(t.call("create_file", {"path": "tests/test_zip.py", "content": "import unittest"}))

## 6. The agent

The new-price task with the rules checked at the end and when acting. Look for the gate's message with R1 and for the rejected edit.

In [ ]:
def agent(tok, model, task, instructions, gate="log", on_edit=False,
          require_marker=False, max_steps=MAX_STEPS, verbose=False):
    """The agent loop with some instructions and a gate: "log", the one of chapter
    1; "rules", which checks the rules on the diff and returns the broken ones; or
    "outcome", which only returns the outcome rules (R4, R7 and R8). With
    "on_edit", the process rules are checked before each edit. With
    "require_marker", an answer without tools that starts with neither DONE nor NOT
    DONE does not end the task: the agent goes on. Returns the row of results, with
    the broken rules at the end."""
    start = time.time()
    messages = [{"role": "system", "content": SYSTEM + "\n\n" + instructions},
                {"role": "user", "content": task["request"]}]
    final, steps, n_in, n_out, rejections = None, 0, 0, 0, 0
    with rt.Workspace() as workspace:
        tools = rt.Tools(workspace)
        if on_edit:
            tools = ToolsWithRules(tools, task)
        for steps in range(1, max_steps + 1):
            text, trimmed = trim_conversation(tok, messages)
            if trimmed is None:
                break
            answer, n_input, n_output = generate(tok, model, text)
            n_in, n_out = n_in + n_input, n_out + n_output
            calls = read_calls(answer)
            if not calls:
                messages.append({"role": "assistant", "content": answer})
                if verbose:
                    print(f"[{steps}] {answer}")
                if require_marker and "DONE" not in answer:
                    messages.append({"role": "user", "content":
                                     "Keep going with the tools, or answer starting "
                                     "with \"DONE:\" or \"NOT DONE:\"."})
                    continue
                if says_done(answer) and rejections < MAX_REJECTIONS:
                    if gate in ("rules", "outcome"):
                        changes = rt.differences(workspace.source, workspace.folder)
                        broken = check_rules(task, tools.log, changes,
                                             answer)
                        if gate == "outcome":   # only the outcome rules
                            broken = {r: text for r, text in broken.items()
                                      if r in ("R4", "R7", "R8")}
                        missing = [f"{r}: {text}" for r, text in sorted(broken.items())]
                    else:
                        missing = whats_missing(tools.log)
                    if missing:
                        rejections += 1
                        notice = ("It isn't done yet according to AGENTS.md: "
                                  + "; ".join(missing) + ". Keep working.")
                        messages.append({"role": "user", "content": notice})
                        if verbose:
                            print(f"    GATE: {notice}")
                        continue
                final = answer
                break
            thought = answer.split("<tool_call>")[0].strip()
            messages.append({"role": "assistant", "content": thought,
                             "tool_calls": [{"type": "function", "function": c}
                                            for c in calls if c["name"]]})
            for call in calls:
                if call["name"] is None:
                    result = {"error": "The tool call cannot be understood."}
                else:
                    result = tools.call(call["name"], call["arguments"])
                if verbose:
                    print(f"[{steps}] {call['name']}"
                          f"({json.dumps(call['arguments'], ensure_ascii=False)[:90]})"
                          f" -> {json.dumps(result, ensure_ascii=False)[:110]}")
                messages.append({"role": "tool", "name": call["name"] or "?",
                                 "content": json.dumps(result, ensure_ascii=False)})
        evaluation = rt.evaluate_attempt(workspace, task)
        changes = rt.differences(workspace.source, workspace.folder)
        broken = check_rules(task, tools.log, changes, final)
    done = final is not None and says_done(final)
    return {"task": task["id"], "type": task["meta"]["type"],
            "solved": evaluation["solved"], "says_done": done,
            "false_done": done and not evaluation["solved"],
            "broken_rules": broken, "rejections": rejections,
            "hidden": evaluation["hidden"], "visible": evaluation["visible"],
            "out_of_scope": evaluation["out_of_scope"],
            "steps": steps, "input_tokens": n_in, "output_tokens": n_out,
            "seconds": round(time.time() - start, 1),
            "answer": final or "(no final answer)",
            "calls": [[r["tool"], r["arguments"].get("path", ""), r["ok"]]
                      for r in tools.log],
            "diff": evaluation["diff"]}


VARIANTS = {
    "prose": {"instructions": AGENTS_PROSE},
    "rules": {"instructions": AGENTS_RULES},
    "checked rules": {"instructions": AGENTS_RULES, "gate": "rules"},
    "rules on the spot": {"instructions": AGENTS_RULES, "gate": "rules",
                          "on_edit": True},
}

In [ ]:
for name in ("checked rules", "rules on the spot"):
    print(f"--- {name}")
    row = agent(tok, model, rt.read_task("T17"), verbose=True, **VARIANTS[name])
    print("Solved:", row["solved"], "· Broken rules:", row["broken_rules"])

## 7. The fourteen tasks with the four variants

By default a sample is run; with `COMPLETE = True`, the book's whole experiment.

In [ ]:
def summary(rows):
    """The totals of a variant, with the broken rules of each kind."""
    by_rule = {}
    for r in rows:
        for rule in r["broken_rules"]:
            by_rule[rule] = by_rule.get(rule, 0) + 1
    return {"tasks": len(rows),
            "solved": sum(r["solved"] for r in rows),
            "says_done": sum(r["says_done"] for r in rows),
            "false_done": sum(r["false_done"] for r in rows),
            "broken_rules": sum(len(r["broken_rules"]) for r in rows),
            "tasks_without_broken_rules": sum(not r["broken_rules"] for r in rows),
            "by_rule": dict(sorted(by_rule.items())),
            "steps": sum(r["steps"] for r in rows),
            "input_tokens": sum(r["input_tokens"] for r in rows),
            "seconds": round(sum(r["seconds"] for r in rows), 1)}


def run_all(tok, model, options, tasks=TASKS):
    """One variant with every task: its rows and its summary."""
    rows = []
    for task_id in tasks:
        row = agent(tok, model, rt.read_task(task_id), **options)
        rows.append(row)
        print(f"  {task_id} {'solved' if row['solved'] else 'no    '}"
              f"  {'says done' if row['says_done'] else 'not said '}"
              f"  broken rules {','.join(sorted(row['broken_rules'])) or '-':12s}"
              f"  steps={row['steps']:2d}  {row['seconds']:6.1f} s", flush=True)
    return {"rows": rows, "summary": summary(rows)}

In [ ]:
# With COMPLETE = False one task is run (about 35 minutes on Colab's T4);
# with True, the chapter's whole experiment (about eighteen hours on the T4).
COMPLETE = False
tasks = TASKS if COMPLETE else ["T17"]
results = {}
for name, options in VARIANTS.items():
    print(f"== {name}")
    results[name] = run_all(tok, model, options, tasks=tasks)
    print(results[name]["summary"])

The book's full results, with every task, are saved in `data/results/chapter_2/`, which `prepare` has downloaded:

In [ ]:
import json
saved = json.loads((ROOT / "data" / "results" / "chapter_2" / "results.json").read_text(encoding="utf-8"))
for name, r in saved.items():
    print(name, r["summary"])

## Exercises

The results files the exercises cite are in `data/results/chapter_2/`, which `prepare` downloads with the data.

1. For each of the rules R1 to R8, say whether it is a process or an outcome rule, which source the evidence comes from (diff, log or answer) and at what moment it can be checked. *(No GPU.)*

2. Read the trace of the new-price task (T17) with the rules checked at the end. Rewrite the message the gate sends the agent about R1 so that it does not push it into touching `config.py` again. Is it enough to change the message, or does the moment when the rule is checked have to change? *(GPU, about 15 minutes.)*

3. Use the prose `AGENTS.md` with the rules checked when acting and at the gate. If the rules are checked, does it still matter how they are written? The results with the fourteen tasks are already in `results_exercises.json`; to see it for yourself, three are enough, for example T04, T10 and T17. *(No GPU with the saved results; GPU, about 30 minutes, with those tasks.)*

4. Write a rule R9, *do not change the signature of an existing function*, and its check with the diff: look for removed and added `def` lines with the same name and different parameters. Is it a process or an outcome rule? Could it be checked when editing? *(No GPU.)*

5. Find the `AGENTS.md` of three open-source projects. Copy five instructions from each one and classify them as checkable or not checkable. For two of the ones that are not checkable, propose a version that is. *(No GPU.)*

6. Make the final gate check only the outcome rules (R4, R7 and R8), without checking anything when editing; in the program it is the option `gate="outcome"`. Does it avoid the trap of the new-price task? How many process rules are broken? The results with the fourteen tasks are already in `results_exercises.json`; to see it for yourself, three are enough, for example T04, T10 and T17. *(No GPU with the saved results; GPU, about 30 minutes, with those tasks.)*

7. Apply the `ASKS_DOCS` expression to the twenty requests of the bank and compare the result with the allowed paths of each contract, in `data/tasks/Txx/contract.json`. Is there any case where they do not match? Who is right? *(No GPU.)*

8. With the rules checked when acting, turn on `require_marker=True`: an answer without tools that says neither "DONE" nor "NOT DONE" no longer ends the task, and the agent goes on. How many tasks does it now claim as done? How many of them are? The results with the fourteen tasks are already in `results_exercises.json`; to see it for yourself, three are enough, for example T03, T10 and T17. *(No GPU with the saved results; GPU, about 45 minutes, with those tasks.)*

9. Count the tokens of the two `AGENTS.md` with the model's tokenizer and multiply them by the number of model calls of each variant. What part of the difference in tokens between the prose and the rules is due to the `AGENTS.md` itself? *(No GPU.)*

10. **Project.** If you use a commercial coding agent that supports hooks, such as Claude Code, write a hook that runs before each edit and blocks it if the file is in a forbidden folder or has not been read earlier in the session. If not, add to `ToolsWithRules` a process rule specific to your repository and test it with three tasks.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). The [AGENTS.md](https://agents.md) convention. Claude Code's hooks: [Hooks reference](https://code.claude.com/docs/en/hooks). The North Cycles store, its program and its tasks are fictional. Original code and experiments.